# 01 - Study areas, model grid and city masks (SRS Phase 1)

Stage A uses **provisional** boundaries from OpenStreetMap. Stage B replaces them with authoritative boundaries verified in QGIS (FR-P1.8). Re-running this notebook with new boundary files regenerates the grid and masks.

In [ ]:
import os, subprocess, sys
REPO_URL = 'https://github.com/Starpredetor/urban-heat-risk-prediction.git'
REPO_DIR = '/content/uhrm'
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR + '/src')
from uhrm.colab import bootstrap
project = bootstrap()

## 1. Find candidate polygons
Search OpenStreetMap and look at the printed areas. Greater Mumbai should be a few hundred km2; a result of about 0 km2 is a building or a point, not the city. Which Navi Mumbai polygon is right is open item OI-1.

In [ ]:
from uhrm import boundaries_osm as osm
navi = osm.search('Navi Mumbai')
osm.show(navi)

In [ ]:
# Greater Mumbai is the union of the Mumbai City and Mumbai Suburban districts.
mumbai = osm.search('Mumbai City district') + osm.search('Mumbai Suburban district')
osm.show(mumbai)

## 2. Choose and save
Set the ids from the lists above. Each saved boundary gets a `.meta.json` recording where it came from (FR-P1.2).

In [ ]:
NAVI_MUMBAI_IDS = [13180880]   # relation found in a first check; confirm it against the list above
MUMBAI_IDS = []                # fill in: the osm_id of each Mumbai district polygon
assert MUMBAI_IDS, 'Set MUMBAI_IDS from the list printed above'
osm.save_boundary(osm.pick(navi, NAVI_MUMBAI_IDS), project.boundary_path(project.area('navi_mumbai')), query='Navi Mumbai')
osm.save_boundary(osm.pick(mumbai, MUMBAI_IDS), project.boundary_path(project.area('mumbai')), query='Mumbai City + Mumbai Suburban districts')

## 3. Model grid, city masks and cell counts

In [ ]:
from uhrm import regions
geoms = {a.slug: regions.load_boundary(project.boundary_path(a), project.grid.crs) for a in project.study_areas}
grid = regions.Grid.covering(list(geoms.values()), res=project.grid.model_resolution_m,
                             buffer_m=project.grid.domain_buffer_km * 1000, crs=project.grid.crs)
masks = {slug: grid.cell_mask(g, project.grid.min_cell_overlap) for slug, g in geoms.items()}
report = regions.cell_report(grid, masks)
for k, v in report.items():
    print(f'{k}: {v}')
regions.save_grid(grid, project.path('grid.json'))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
canvas = np.zeros(grid.shape)
for i, m in enumerate(masks.values(), start=1):
    canvas[m] = i
x0, y0, x1, y1 = grid.bounds
fig, ax = plt.subplots(figsize=(7, 8))
ax.imshow(canvas, extent=(x0, x1, y0, y1), cmap='viridis')
for g in geoms.values():
    for poly in getattr(g, 'geoms', [g]):
        ax.plot(*poly.exterior.xy, 'w-', lw=0.8)
ax.set_title('Model domain: 1 km cells assigned to each city')
plt.show()

## Checks before moving on
- Both outlines sit where the cities are, and the grid is large enough for 64 x 64 patches (OI-3).
- `overlap_between_cities` is 0.
- Note the cell counts; they decide how much statistical power per-city results have.
- Final verification of the boundaries happens in QGIS in Stage B.